In [1]:
import torch
import time

https://docs.pytorch.org/docs/stable/notes/get_start_xpu.html

In [2]:
print(f"Versión de PyTorch: {torch.__version__}")

# 1. Verificación básica
disponible = torch.xpu.is_available()
print(f"¿XPU (GPU Intel) disponible?: {disponible}")

if disponible:
    # 2. Información del hardware detectado por PyTorch
    nombre_gpu = torch.xpu.get_device_name(0)
    print(f"GPU detectada: {nombre_gpu}")
    
    # 3. Prueba de fuego: Operación matemática real en la GPU
    # Creamos dos tensores grandes directamente en la XPU
    a = torch.randn(2000, 2000, device="xpu")
    b = torch.randn(2000, 2000, device="xpu")
    
    # Realizamos una multiplicación de matrices (punto de estrés)
    c = torch.matmul(a, b)
    
    print("Prueba de cálculo completada. ¡El tensor está en la GPU!")
    print(f"Ubicación del resultado: {c.device}")
else:
    print("PyTorch no ve la GPU. Verifica que instalaste con '--index-url https://download.pytorch.org/whl/xpu'")

Versión de PyTorch: 2.11.0+xpu
¿XPU (GPU Intel) disponible?: True
GPU detectada: Intel(R) Arc(TM) Graphics
Prueba de cálculo completada. ¡El tensor está en la GPU!
Ubicación del resultado: xpu:0


In [6]:
def benchmark(device_name, size=8000):
    # Definir el dispositivo
    device = torch.device(device_name)
    print(f"\n🚀 Iniciando benchmark en: {device_name.upper()}")
    
    # Crear dos matrices grandes
    # (8000x8000 es lo suficientemente grande para notar la diferencia)
    try:
        a = torch.randn(size, size, device=device)
        b = torch.randn(size, size, device=device)
        
        # Calentamiento (para que la GPU se despierte)
        _ = torch.matmul(a, b)
        if device_name == "xpu":
            torch.xpu.synchronize()
            
        # Medición real
        start_time = time.time()
        for _ in range(10):
            c = torch.matmul(a, b)
        
        if device_name == "xpu":
            torch.xpu.synchronize() # Esperar a que la GPU termine
            
        end_time = time.time()
        
        avg_time = (end_time - start_time) / 10
        print(f"✅ Tiempo promedio por operación: {avg_time:.4f} segundos")
        return avg_time
    except Exception as e:
        print(f"❌ Error en {device_name}: {e}")
        return None

# Ejecutar comparativa
if __name__ == "__main__":
    print(f"Versión de PyTorch: {torch.__version__}")
    
    t_cpu = benchmark("cpu")
    
    if torch.xpu.is_available():
        t_gpu = benchmark("xpu")
        
        if t_cpu and t_gpu:
            print(f"\n📊 RESULTADO FINAL:")
            print(f"La GPU es {t_cpu / t_gpu:.2f} veces más rápida que la CPU.")
    else:
        print("\n❌ La GPU XPU no está disponible para PyTorch.")

Versión de PyTorch: 2.11.0+xpu

🚀 Iniciando benchmark en: CPU
✅ Tiempo promedio por operación: 2.0076 segundos

🚀 Iniciando benchmark en: XPU
✅ Tiempo promedio por operación: 0.2888 segundos

📊 RESULTADO FINAL:
La GPU es 6.95 veces más rápida que la CPU.
